<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/SemanticKernelLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">3c - Workshop (AI Extensions) - Open Source Decision Intelligence</span>
</div>

Decision Intelligence applied in this module:  
* Listing key factors to consider when making a sound decision  
* Using OSS (open-source) reasoning models to optimize the decision approach 
* Decision Scenario: Use a decision framework (Ben Franklin's Pro & Con List) to create a decision plan  
* Improving Decision Intelligence process by explicitly providing decision frameworks and additional context 

A recommended enterprise pattern is to scale Articial Intelligence strategy with three key pillars:
* **Commercial AI** (OpenAI and other proprietary Generative AI providers)
* **Open Source (OpenWeight) AI** (open-source AI providers)
* **Vendor and Partner AI** (i.e. company HR Software, contract software) 

> 📝 Note: There is a technical difference between Open Source and Open Weight models. Open Weight models generally only provide you the weights & biases file(s) with a friendly open source license. Open Source models include the pre/post training data, training scripts and sometimes the training checkpoints. Those three additions allow the models to be fully retrained by anyone. More info can be found here: https://opensource.org/ai/open-weights. For this workshop, we will use the term open source more loosely. A great majority of the cases most organizations just want two things: the weights & biases file(s) and a friendly licnese. Certain organizations certainly require full model assets for risk management. While the full open source model assets are nice to have, they offer minimal value in most cases. 

These three pillars (listed above) strategically form AI capability and capacity in what I like to refer to as the **"Generative AI Brain"**. This is illustrated below with sample providers. Most organizations scale their AI investments with Commercial AI providers, such as: OpenAI, Anthropic or Google. However, some organizations prefer to have more governance controls over their AI and prefer open-source alternatives. For hobby consumers, open source translates to no credit card requirements for APIs and very high privacy. 
Note that there are some commercial providers like Meta or Cohere that provide open weight models. Finally in the third tier, an organization may have an existing relationship with a vendor/partner that has their own AI integrations. For example, Adobe offers enterprise graphic design/contract software and leveraging their built-in Generative AI capabilities rather building their own can make a lot of sense. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/AIBrainPillars.png">

Most developer AI frameworks work across all the pillars mentioned above. It allows all types of models (commercial or proprietary) and almost any APIs to be orchestrated to faciliate enterprise Decision Intelligence. This is a much deeper subject and for simplicity this module highlights how to use local GenAI models (i.e. LLMs) with Microsoft AI for decision-making. Models from the OpenAI OSS family will be selected to run locally using LMStudio as a local REST endpoint that will interface with Microsoft AI orchestration.  

Open Source (OSS) models vary dramatically in size. They can have a small number of parameters/activated layers and perform great locally on your mobile device! They can also have a huge number parameters that rivals commercial AI LLMs. 

In the cases where OSS models have a small number of parameters, they are are considered SLMs (Small Language Models) with parameters generally below the 30 Billion parameter threshhold. This allows most of these models to run comfortably on commodity hardware available to personal users. This doesn't mean these models are not capable of performing well. While these models certainly may lack the general knowledge breadth of frontier AI Large Language Models, SLMs make up for it by providing very specialized logic, math and reasoning capabilities. For example, an SLM only trained for a specific language (English) and a certain domain (legal industry) can include only the relevant training information. Therefore, it can be offered as a much more compact model with many less parameters. 

In the cases where OSS models have a large number of parameters, they can perform just as well as commercial models! OSS models with large parameters require enterprise commmerical hardware to execute at scale. Below is an image from an independent AI benchmarking site ArtificialAnalysis.ai across proprietary and OSS models. Notice that almost half of the Top 30 performing models are open weight models (blue): 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/ArtificialAnalysis-ModelBenchmark-202604.png">  

For this workshop, we will be working primarily with SLM family of OSS models, because not everyone has H100 or B200 Nvidia GPUs to deploy huge parameter models. However, as you will see these smaller models are quite good. As of April 2026, OpenAI's gpt-oss-20 or Gemma 4 open-source reasoning models performs about 2 generations back of frontier LLM performance. Keep in mind these are general 20B & 26B parameter OSS models. This model can be fine-tuned or mid-trained and can perform much better. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/LMStats-Benchmark-202604.png">  

----
### Step 1 - Get Started with LMStudio and Local Open Source AI Models 

Steps to get started:
* Download & install the latest LMStudio version: https://lmstudio.ai/ (Windows, Mac or Linux) 
* Run the LMStudio studio application.
* In the LMStudio application, search for **Gemma 4 26B A4B** GGUF (MLX is optimized on macOS) in the "Discover" section of LMStudio. A variety of Gemma options that are official and unofficial from hobbyists will appear. Typically, selecting the official model with the most downloads will provide the best results. In this case, you can be safe by selecting the official **Google** provider. You can select different quantizations of the model, to optimize the performance. 
* In the experiment below, the **26B parameter model is being used with 4bit quantization** is selected. LMStudio will inspect your hardware and let you know which quantized version of the model(s) is optimal for your hardware. Note: Even computers with small graphics cards can run these models well locally. Furthermore, laptops such as the Macbook Pro with Neural Engine can run LMStudio local models as well. You just need memory. 
* Start the LMStudio Server with the **Gemma 4 26B A4B** model loaded. This will start a local REST endpoint with a URI similar to http://10.0.0.18:1234/v1 
* The LMStudio local server does not have default security, you can simply check by navigating to this link in any browser to check if a model is loaded: **http://10.0.0.18:1234/v1/models** in the web browser. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/LMStudioServer.png"> 

----
### Step 2 - Initialize ChatClient using OpenAI libraries

Execute the next cell to:
* Use the Configuration Builder to use the local LMStudio Server  
* Use the local API configuration to build an API compatible OpenAIClient
* The API Compatible OpenAIClient can be converted to a Microsoft.Extensions.AI ChatClient abtraction
* Note: Notice there is no security being passed in and it is simply a URL

In [4]:
// Install the required AI packages
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.6"
#r "nuget: Microsoft.Extensions.AI, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.5.0"
#r "nuget: OpenAI, 2.10.0"

using Microsoft.Extensions.AI;
using OpenAI;
using System.ClientModel; // used by APiKeyCredential class

// Note: Ollama (you can use this if wanting to use Ollama locally with MEAI extensions)
// IChatClient chatClient = new OllamaApiClient(new Uri("http://127.0.0.1:11434"), "gpt-oss:20b");

var apiKey = "not_needed_for_lmstudio_ollama_etc"; // local LLMs do not require an API key
var localUrl = "http://10.0.0.61:1234/v1/"; // this can be localhost or an IP address
var localModelName = "google/gemma-4-26b-a4b"; // Another Option: "openai/gpt-oss-20b";

var apiCredentials = new ApiKeyCredential(apiKey);
var openAIClientOptions = new OpenAIClientOptions
{
    Endpoint = new Uri(localUrl)
};

// Create a local AI client 
var localAIClient = new OpenAIClient(apiCredentials, openAIClientOptions);

// Wrap the OpenAI-compatible chat client with the Microsoft.Extensions.AI abstraction.
IChatClient localAIChatClient = localAIClient
    .GetChatClient(localModelName)
    .AsIChatClient();

Installed Packages Microsoft.Extensions.AI, 10.5.0 Microsoft.Extensions.AI.Abstractions, 10.5.0 Microsoft.Extensions.AI.OpenAI, 10.5.0 Microsoft.Extensions.DependencyInjection, 10.0.6 OpenAI, 2.10.0

---
### Step 3 - Open Source AI with Decision Intelligence 

The OpenAI .NET library allows one to interact with any API service that adheres to the OpenAI specification. Notice the method to add LMStudio capability was simply enabled via the **AddOpenAIChatCompletion** method above. 

Execute the cell below about decision factors for a investment property. Note:
* OpenAI Prompt Execution Settings are the same in LMStudio as they are for OpenAI and Azure OpenAI
* OSS models have specific model cards identifying best practices 
* OSS reasoning models will output their reasoning tokens inside the <think> XML tags
* Passing in arguents works the same way in OpenAI as other models (i.e. Temperature, Reasoning Effort etc.)

> 📝 **Note:** It is important to note that different model providers (commercial and open-source) have different recommended general settings that have been tested. While different types of AI systems will require optimizing these settings, recommended settings are a good starting point.

In [5]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Provide detailed decision-making criteria for evaluating the investment decision
var simpleDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

What key factors should you evaluate to ensure a sound investment decision, including financial, 
market, and property-specific considerations? 
Outline the critical steps and criteria for assessing location, potential rental income, 
financing options, long-term property value, and associated risks. 
""";

List<ChatMessage> chatMessages =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, simpleDecisionPrompt),
];

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryResponse = await localAIChatClient.GetResponseAsync(chatMessages);
var chatHistoryResponseText = chatHistoryResponse.Text;

// Display the response string as Markdown
chatHistoryResponseText.DisplayAs("text/markdown");

| Dimension | Key Evaluation Criteria | Critical Steps & Assessment Process | Risk Management Strategy |
| :--- | :--- | :--- | :--- |
| **Financial Fundamentals** | ROI, Cash-on-Cash Return, Tax Implications (Depreciation/Deductions), Entry/Exit costs. | Perform sensitivity analysis on interest rates; consult a tax professional to optimize deductions; calculate Net Operating Income (NOI) and break-even points. | **Risk:** Negative cash flow or inflation erosion. <br>**Mitigation:** Maintain a robust cash reserve and use conservative valuation models. |
| **Market Dynamics** | Local Supply/Demand, Economic growth (job markets), Demographic shifts. | Analyze historical price trends; research local development/infrastructure projects; study absorption rates and vacancy trends in the area. | **Risk:** Economic downturn or market saturation. <br>**Mitigation:** Invest in areas with diversified employment bases and growing population influx. |
| **Location & Property Logistics** | Proximity to transit/amenities, Neighborhood safety, Physical condition/structural integrity. | Conduct thorough site visits; check zoning laws and land-use regulations; perform professional inspections of all core systems (HVAC, Roof, Foundation). | **Risk:** Structural failures or neighborhood degradation. <br>**Mitigation:** Prioritize properties in high-demand locations and conduct rigorous due diligence before purchase. |
| **Income & Occupancy Potential** | Rental yield, Seasonality (if vacation home), Regulatory compliance. | Compare property against local rental comparables; verify legality of short-term/long-term rentals with municipal codes. | **Risk:** Regulatory changes (e.g., STR bans) or seasonal vacancy periods. <br>**Mitigation:** Verify legal loopholes and plan for multi-use occupancy (e.g., long-term vs short-term). |
| **Long-term Strategy & Financing** | Appreciation potential, Mortgage terms for second homes, Liquidity/Exit strategy. | Evaluate LTV (Loan-to-Value) requirements for non-primary residences; determine the timeline and feasibility of an exit (sale vs. refinance). | **Risk:** Interest rate volatility and asset illiquidity. <br>**Mitigation:** Secure stable, fixed-rate financing and target high-appreciation zones. |

---
### Step 4 - Open Source AI with Decision Intelligence (Advanced)

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-WillRodgers-RealEstate.png"> 

Advanced Prompt Engineering techniques can be applied to OSS (open-source) models as well. In the example below a more advanced reasoning decision prompt will be used to provide additional instructions to the GenAI model. Reasoning models do a nice job in approaching the problem with an inner monologue, however you can provide additional instructions for the model to consider as they are thinking about an approach.  

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Use Chain of Thought to prompt the OSS model
// Use the Minto Pyramid to communicate the decision 
var advancedDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

Before providing any answer, in your reasoning process consider the following:
Understand the Problem: Carefully read and understand the user's question or request. 
Break Down the Reasoning Process: Outline the steps required to solve the problem or respond to the request logically and sequentially. Think aloud and describe each step in detail. 
Always aim to make your thought process transparent and logical. 
Explain Each Step: Provide reasoning or calculations for each step, explaining how you arrive at each part of your answer. 
Provide structured, logical, and comprehensive advice. 
Arrive at the Final Answer: Only after completing all steps, provide the final answer or solution. 
Review the Thought Process: Double-check the reasoning for errors or gaps before finalizing your response. 
Communicate the final decision using the Minto Pyramid Principle.
""";

List<ChatMessage> chatMessagesDecision =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, advancedDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64,
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryAdvancedDecisionPromptResponse = await localAIChatClient.GetResponseAsync(chatMessagesDecision, chatOptions);
var chatHistoryAdvancedDecisionPromptText = chatHistoryAdvancedDecisionPromptResponse.Text;

// Render the response string as Markdown
chatHistoryAdvancedDecisionPromptText.DisplayAs("text/markdown");

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatMessages.Add(new ChatMessage(ChatRole.Assistant,chatHistoryAdvancedDecisionPromptText));

| Phase | Action Item | Description | Strategic Value |
| :--- | :--- | :--- | :--- |
| **1. Financial Feasibility Analysis** | Capital Requirement & Cash Flow | Calculate total acquisition costs (down payment, closing costs, taxes) vs. projected net rental income after maintenance and vacancy. | Determines if the asset is self-sustaining or a capital drain. |
| **2. Market Intelligence** | Local Macro & Micro Trends | Evaluate neighborhood growth, employment rates, tourism trends (if short-term rental), and school district quality. | Mitigates the risk of purchasing in a declining or stagnant market. |
| **3. Risk & Sensitivity Assessment** | Scenario Planning | Model "Best Case," "Expected," and "Worst Case" scenarios (e.g., interest rate hikes, prolonged vacancy). | Identifies the break-even point and potential for loss. |
| **4. Regulatory & Legal Audit** | Compliance Check | Verify zoning laws, HOA restrictions, short-term rental (STR) legality, and tax implications for secondary residences. | Prevents legal entanglement and sudden loss of utility. |
| **5. Opportunity Cost Evaluation** | Alternative Investment Comparison | Compare the expected ROI of the property against passive index funds or other liquid assets. | Ensures capital is being deployed in its most productive manner. |
| **Decision Framework** | **The Final Verdict** | **Proceed only if: 1. Net Cash Flow is positive in the 'Worst Case' scenario; 2. The asset meets specific growth/usage criteria; 3. Risk is diversified from your primary residence.** | **A structured 'Go/No-Go' decision based on quantitative and qualitative data.** |

---
### Step 5 - Open Source AI with The Ben Franklin Decision Framework 

> 📜 **_"By failing to prepare, you are preparing to fail."_**
>
> -- <cite>Ben Franklin (Founding Father of the United States, inventor, godfather of Decision Science)</cite> 

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/Articles/main/20230326-Make-Great-Decisions-Using-Ben-Franklins-Pros-And-Cons-Method/Image-BenFranklinDecisionMakingMethod.png">

#### Tom Brady's use of a Decision Framework

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-TomBrady-ProsAndConsList.png">

Tom Brady's decision to join the Tampa Bay Buccaneers in 2020 marked a significant in his legendary NFL career. After 20 seasons and six Super Bowl championships with the New England Patriots, Brady became a free agent and chose to sign with the Bucs. How did he arrive at this decision? On the Fox broadcast on 09.29.2024, while covering the Buccaneers vs Philadelphia Eagles game, Tom Brady described how he arrived at this decision.

In the screenshot below, Tom Brady is holding up some small paper cards he is showing the audience of the broadcast. Brady mentioned he wrote down the personal decision criteria that was important and how each team compared in that criteria (salary, weather etc). He used this to select the Tampa Bay Buccaneers as his team, where he went on to win a Super Bowl in his first year there! **After 250 years since it's inception, Tom Brady used the "Ben Franklin Decision Framework" to decide where to play NFL quaterback!!**  

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/OpenSourceDecisionIntelligence-BenFranklinDecisionFramework-TomBrady.png">

#### Steps for Ben Franklin's Decision Framework

Below are the steps Ben Franklin recommends when making a decision, which he called his "Decision Making Method of Moral Algebra":  
- Frame a decision that has two options (Yes or a No)
- Divide an area into two competing halves: a "Pro" side and "Con" side
- Label the top of one side "Pro" (for) and the other "Con" (against)
- Under each respective side, over a period of time (Ben Franklin recommended days, this could be minutes) write down various reasons/arguments that support (Pro) or are against (Con) the decision
- After spending some time thinking exhaustively and writing down the reasons, weight the different Pro and Con reasons/arguments
- Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
- The side with the most remaining reasons is the option one should select for the decision in question

Learn more about Ben Franklin's Decision Framework: https://medium.com/@bartczernicki/make-great-decisions-using-ben-franklins-decision-making-method-c7fb8b17905c  

#### Decision Scenario - Should a Family Take a Luxury Vacation

Should a family take a luxury family vacation this year? Just like Brady mapped out whether joining the Bucs would satisfy his key personal and professional goals, you can list the factors that matter most for your family—budget, timing, destination climate, activities for the kids—and lay them out on your own “decision cards.” Weigh each component carefully, just as Brady weighed his NFL future. Because if it worked to land Brady in Tampa (where he won yet another Super Bowl), imagine what it can do for a family’s dream getaway.

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-Vacation.png">

In [11]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making. 

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";
var benFranklinLuxuryVacationDecisionPrompt = """
Apply the Ben Franklin Decision-Making Framework (Pro and Con list) to evaluate whether or not to take a luxury family vacation. 
List at most 5 pros and at most 5 cons to help the user make an informed decision.
""";

List<ChatMessage> chatMessagesLuxuryVacationDecision =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, benFranklinLuxuryVacationDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryAdvancedDecisionPromptResponse = await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecision, chatOptions);
var chatHistoryAdvancedDecisionPromptText = chatHistoryAdvancedDecisionPromptResponse.Text;

// Render the response string as Markdown
chatHistoryAdvancedDecisionPromptText.DisplayAs("text/markdown");

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatMessages.Add(new ChatMessage(ChatRole.Assistant,chatHistoryAdvancedDecisionPromptText));

| Category | Factor | Description | Potential Impact |
| :--- | :--- | :--- | :--- |
| **Pro** | Family Bonding | Quality time away from daily stressors and work obligations. | Strengthens emotional connections and creates lifelong memories. |
| **Pro** | Mental Recharge | Significant break from routine allows for mental decompression and relaxation. | Increases long-term productivity and reduces burnout symptoms. |
| **Pro** | Cultural Enrichment | Exposure to new environments, cuisines, and educational experiences. | Broadens perspectives and provides unique learning opportunities for children. |
| **Pro** | Reward/Milestone | Celebrating a significant life achievement or successful year. | Validates hard work and provides a sense of accomplishment. |
| **Pro** | Relationship Building | Shared experiences can resolve underlying tensions through positive interaction. | Improves family cohesion and interpersonal dynamics. |
| **Con** | Financial Strain | High immediate cost may impact savings or debt-repayment goals. | Could lead to short-term financial stress or long-term budget constraints. |
| **Con** | Opportunity Cost | Funds spent on luxury travel cannot be invested in other assets (e.g., education, house). | Reduces the capital available for future strategic life goals. |
| **Con** | Logistical Stress | Travel planning, transit, and itinerary management can be taxing. | May lead to friction or exhaustion rather than relaxation. |
| **Con** | Post-Vacation Blues | Difficulty reintegrating into daily reality after an extreme contrast in lifestyle. | Potential dip in morale or motivation upon returning to routine. |
| **Con** | Disturbance of Routine | Disruption of sleep schedules, diet, and work responsibilities. | May result in physical fatigue or backlog of tasks upon return. |

#### Improving the Ben Franklin's Decision Framework with SLMs

For those familiar with the Ben Franklin decision framework, the output from the AI model above may not be exactly what most would anticipate. The Ben Franklin framework could be partially understood by the AI process nor fully applied. Open-Source GenAI models that have a small amount of parameters (< ~27 billion parameters) may not have all the inherent Decision Intelligence knowledge "trained" into the model. The exception being domain-specific models that are specifically trained on data sets for that domain. These domain-specific models can fill their "limited knowledge" with information that is pertinent to the tasks, while maintaining a small amount of parameters. Therefore, you could train small AI models that specialize in Decision Intelligence. 

One simple way to improve the outcome is to provide the explicit steps of the "Ben Franklin Decision Framework" into the prompt context. This basically provides the instructions of the decision framework directly to the model; regardless if the GenAI model was trained with decision framework data. By doing this extra explicit step, there is no ambiguity for the AI model how to approach the decision process. 

In the example below, the prompt context is provided with the Ben Franklin Decision Framework steps. Contrast this with the example above, where the decision recommendation is not clear. 

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option. If there less than 5 reasons, only list the actual number of reasons there are for each side. Do not list filler reasons to reach 5 if there are not actually 5 reasons for that side.
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question

IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Question: Should I take a luxury family vacation?
""";

List<ChatMessage> chatMessagesLuxuryVacationDecisionImproved =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, explicitBenFranklinDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedResponse = await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImproved, chatOptions);
var chatLuxuryVacationDecisionImprovedResponseText = chatLuxuryVacationDecisionImprovedResponse.Text;

// Render the response string as Markdown
chatLuxuryVacationDecisionImprovedResponseText.DisplayAs("text/markdown");

| Step | Description / Application | Details |
| :--- | :--- | :--- |
| 1. Decision Frame | Decision: Should I take a luxury family vacation? | Option A: Yes / Option B: No |
| 2-4. Argument Listing | Pro (For) | 1. Strengthens family bonds through shared experiences.<br>2. Creates lifelong memories for children.<br>3. Provides high-quality relaxation and stress relief.<br>4. Reward for hard work and recent achievements.<br>5. Exposure to new cultures and unique environments. |
| | Con (Against) | 1. Significant financial strain or depletion of savings.<br>2. Logistical stress of planning and travel.<br>3. Potential for conflict in close quarters.<br>4. Opportunity cost of not investing money elsewhere.<br>5. Post-vacation "re-entry" blues or fatigue. |
| 5-6. Weighting and Balancing | Evaluation Process | Let's group the arguments to determine relative weight.<br><br>**Weighting Group 1 (High Value):**<br>Pro: Creating lifelong memories (Reason 2) + Strengthening bonds (Reason 1) = 1 High Value Pro.<br>Con: Significant financial strain (Reason 1) = 1 High Value Con.<br>*Result: These balance out and are removed from the count.*<br><br>**Weighting Group 2 (Medium Value):**<br>Pro: Relaxation/Stress relief (Reason 3) + Reward for hard work (Reason 4) = 1 Medium Value Pro.<br>Con: Opportunity cost/Investment loss (Reason 4) = 1 Medium Value Con.<br>*Result: These balance out and are removed from the count.*<br><br>**Weighting Group 3 (Low/Remaining Value):**<br>Pro: Exposure to cultures (Reason 5) = 1 Low Value Pro.<br>Con: Logistical stress (Reason 2) + Conflict potential (Reason 3) + Re-entry fatigue (Reason 5) = 1 Low Value Con.<br>*Note: Since the goal is to find which side has more remaining reasons, we look at the final count.* |
| 7. Final Decision | Remaining Reasons Count | **Remaining Pro Reasons: 0**<br>**Remaining Con Reasons: 0**<br><br>Wait, let's re-apply step 7 strictly based on the raw count after weight balancing. If we look at unique points that were not able to be perfectly paired by value:<br><br>If we assume 1 high-value pro cancels 1 high-value con, and 2 medium pros cancel 1 medium con...<br><br>Let's re-evaluate the count:<br>Pro: 5 reasons.<br>Con: 5 reasons.<br><br>If we pair them:<br>(Pro 1+2) vs (Con 1) -> [Balance]<br>(Pro 3+4) vs (Con 4) -> [Balance]<br>Remaining: Pro 5 vs Con 2,3,5.<br><br>Wait—If the Con side has more remaining reasons (2, 3, and 5) after the heavy weights are balanced against the Pro side, then according to rule #7: **The decision is NO.** |
| **Final Recommendation** | **Decision: No** | **Reasoning: Based on the Ben Franklin framework applied, after weighting and balancing high-impact reasons, the Con side retains more individual arguments. Therefore, you should not take the luxury family vacation.** |

The GenAI model may or may not recommend a luxury vacation depending on the executed run. It's decision response is highly generic and isn't grounded on personal information that can influence the decision recommendation. This can be dramatically improved further! Imagine if the GenAI model had access to: your finances, current stress level, the last time you took a vacation, any upcoming major purchases, family dynamic?!  

In the optimized decision example below, additional context is provided with that information. Notice how it changes the the Pro and Con list.  

Just like Tom Brady, the AI could craft a Pro and Con list specific and personalized to your scenario!

In [19]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";
// Try changing the background information to see how it affects the decision-making process
var backroundInformation = """
BACKGROUND INFORMATION OF FAMILY: 
You are considering purchasing a secondary home as an investment property. 
You have been stressed out at work. 
You have been working long hours and have not taken a vacation in over a year. 
You have received a recent promotion (pay raise with a large bonus coming in a few months).
Your car is finishing its lease and will need to be replaced soon. 
""";
// Try to adjust the specificity of the decision-making criteria to see how it affects the decision-making process
var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option. If there less than 5 reasons, only list the actual number of reasons there are for each side. Do not list filler reasons to reach 5 if there are not actually 5 reasons for that side.
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question

IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Question: Should I take a luxury family vacation?
""";

List<ChatMessage> chatMessagesLuxuryVacationDecisionImprovedWithBackground =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, backroundInformation),
    new(ChatRole.User, explicitBenFranklinDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedWithBackgroundResponse = 
    await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImprovedWithBackground, chatOptions);
var chatLuxuryVacationDecisionImprovedWithBackgroundResponseText = chatLuxuryVacationDecisionImprovedWithBackgroundResponse.Text;

// Render the response string as Markdown
chatLuxuryVacationDecisionImprovedWithBackgroundResponseText.DisplayAs("text/markdown");

| Step | Application to Decision | Details |
| :--- | :--- | :--- |
| 1) Frame the Decision | Should I take a luxury family vacation? | Option A: Yes |
| | | Option B: No |
| 2, 3, 4) List Arguments | Pro (For) | 1. Mental health relief from work stress |
| | | 2. Reward for recent promotion and hard work |
| | | 3. Necessary rest after one year without vacation |
| | Con (Against) | 1. High cost impacting car replacement funds |
| | | 2. Potential conflict with upcoming investment purchase |
| 5, 6) Weighting and Importance | Analysis of weight/importance | The three "Pro" reasons (Mental health, Reward, Rest) are combined into one significant value: Personal Well-being. |
| | | The two "Con" reasons (Car funds, Investment) are combined into one significant value: Financial Constraint. |
| | Comparison of remaining reasons | Pro Side Remaining: 1 (Personal Well-being) |
| | | Con Side Remaining: 1 (Financial Constraint) |
| 7) Final Decision | Determination of choice | Because the Pro side contains multiple distinct reasons that contribute to a single weight, and even when weighted against the Cons, we must look at the remaining count of arguments. In this specific logical application where weights are balanced, if we evaluate based on the density of original points provided: |
| | | **Decision: Yes** (The Pro side originally had 3 reasons to the Con's 2, providing more substance to the decision). |

Notice how providing personal family background information changes the entire dynamic of the information used in the decision framework and how it influences the recommended decision. The decision process is more specific not only to the scenario, but also it provides contextual background information. This makes the decision process more personalized and potentially much more accurate! 

---
### Step 6 - Open Source AI & Commercial AI Providers  

Semantic Kernel can include mutliple AI service providers. This allows for hybrid workflows from a single Semantic Kernel instance like: 
* Capability Optimizations: Use SLMs for domain specific tasks and LLMs for complex decision reasoning
* Decision Optimizations: Apply an (ensemble) decision self-consitency pattern  
* Capacity Optimizations: Splitting functions, plugins, personas or agents across different AI services

In [ ]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 9.0.6"
#r "nuget: Microsoft.Extensions.Configuration.Json, 9.0.6"
#r "nuget: Microsoft.SemanticKernel, 1.60.0"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using Microsoft.SemanticKernel.Connectors.OpenAI;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// Retrieve the configuration settings for the Azure OpenAI service
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];

// Example to build a Kernel with Azure OpenAI and Opensource AI
var semanticKernel = Kernel.CreateBuilder()
    .AddAzureOpenAIChatCompletion(
        modelId: azureOpenAIModelDeploymentName,
        deploymentName: azureOpenAIModelDeploymentName,
        endpoint: azureOpenAIEndpoint,
        apiKey: azureOpenAIAPIKey)
    .AddOpenAIChatCompletion(
        modelId: "Phi-4-Reasoning",
        endpoint: new Uri("http://localhost:1234/v1/"),
        apiKey: "LMStudio")
    .Build();

Installed Packages Microsoft.Extensions.Configuration, 9.0.6 Microsoft.Extensions.Configuration.Json, 9.0.6 Microsoft.SemanticKernel, 1.60.0